# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import pandas as pd
from pathlib import Path

# Locate and load the dataset (supports both processed feature vector and raw parquet)
processed_path = Path("../data/processed/refresh_feature_vector.csv")
raw_path = Path("../data/raw/content_refresh_3m.parquet")
df = pd.read_csv(processed_path)
source_name = "Processed Feature Vector"

## 1. Build the feature vector

To build the feature vector efficiently and maintain clean, modular code, the core data transformations are implemented via a structured pipeline (`01_prepare_features.py`)[cite: 1]. 

The pipeline processes raw search data sequentially[cite: 1]:
1. **Data Cleaning & Filtering:** Removes rows with insufficient visibility (`total_impressions < 100`), drops duplicates based on `content_hash_id`, and handles missing numeric/categorical values safely[cite: 1].
2. **Sequential Feature Transformations:** Applies modular functions in a strict logical order—starting with binary flags (`add_has_flags`), moving to base content and market attributes, deriving freshness and age dynamics, creating SERP position buckets (`add_position_bucket`), computing relative peer-group metrics, and finally calculating compound interaction features[cite: 1].
3. **Leakage Prevention:** Explicitly excludes outcome labels and future performance metrics (such as clicks, sessions, or CTR) from the feature set.

In [7]:
df_features = pd.read_csv(processed_path)
print(f"\n Final Feature Vector Dimensions: {df_features.shape[0]:,} rows × {df_features.shape[1]} columns")
print("\n Sample of engineered feature columns preview:")
sample_preview_cols = ["word_count", "content_depth", "freshness_score", "keyword_opportunity", "pos_bucket"]
display(df_features[[c for c in sample_preview_cols if c in df_features.columns]].head(3))


 Final Feature Vector Dimensions: 118,092 rows × 84 columns

 Sample of engineered feature columns preview:


,word_count,content_depth,freshness_score,keyword_opportunity,pos_bucket
0,0,0.000000,1.0,71.428571,51-100
1,2406,7.786136,1.0,1000.000000,21-50
2,2826,7.946971,1.0,1000.000000,21-50


## 2. Feature notes (meaning, missing, categorical, available-when?)

The table below documents key features engineered through our pipeline[cite: 1], their missing value policies, and verifies that they are strictly available **before** the prediction moment (preventing timeline leakage):

| Feature Name | Meaning & Description | Missing Value Policy | Available Before Prediction? |
| :--- | :--- | :--- | :--- |
| `word_count` | Total word count of the content page. | Filled with `0` via pipeline[cite: 1]. | **Yes** (Known at publication/audit time). |
| `search_volume` | Estimated monthly search volume for the target keyword. | Filled with `0` via pipeline[cite: 1]. | **Yes** (Historical SEO market data). |
| `content_age_days` | Number of days since the content was first published. | Retained as numeric bounds. | **Yes** (Calculated from historical timestamps). |
| `days_since_update` | Time elapsed since the last content modification. | Clipped to lower bound 0, filled[cite: 1]. | **Yes** (Content management history). |
| `freshness_score` | Inverse decay score measuring content update recency ($1 / (1 + \text{days})$)[cite: 1]. | Imputed via default cap (365 days)[cite: 1]. | **Yes** (Derived from historical update logs). |
| `pos_bucket` | Categorical SERP position range group (e.g., `1-3`, `4-5`)[cite: 1]. | Binned via `pd.cut`[cite: 1]. | **Yes** (Baseline position before outcome window). |
| `keyword_opportunity` | Ratio of search volume to market competition[cite: 1]. | Handled by replacing zero competition with `0.01`[cite: 1]. | **Yes** (Derived from market inputs). |

In [8]:
# Programmatic audit of feature characteristics, data types, and missing states
print(f"{'='*60}")
print("FEATURE NOTES AUDIT & DATA TYPE CHECK")
print(f"{'='*60}")

# Select key analytical features engineered by the script
audit_features = [
    "word_count", "search_volume", "content_age_days", 
    "days_since_update", "freshness_score", "keyword_opportunity"
]

existing_audit_cols = [c for c in audit_features if c in df_features.columns]

# Display summary statistics and data types
summary_df = pd.DataFrame({
    "Data Type": df_features[existing_audit_cols].dtypes,
    "Missing Count": df_features[existing_audit_cols].isnull().sum(),
    "Non-Zero Min": [df_features[c].min() for c in existing_audit_cols],
    "Max": [df_features[c].max() for c in existing_audit_cols]
})

print(summary_df.to_string())
print("\n Feature notes and pipeline bounds verified successfully.")

FEATURE NOTES AUDIT & DATA TYPE CHECK
                    Data Type  Missing Count  Non-Zero Min        Max
word_count              int64              0      0.000000    29341.0
search_volume           int64              0      0.000000   246000.0
content_age_days        int64              0      1.000000      494.0
days_since_update       int64              0      0.000000      303.0
freshness_score       float64              0      0.003289        1.0
keyword_opportunity   float64              0      0.000000  7400000.0

 Feature notes and pipeline bounds verified successfully.


## 3. The leakage hunt

To ensure the model learns generalizable patterns rather than memorizing answers through data leakage, we perform a rigorous leakage hunt across three critical risk categories:

1. Label-Derived Columns: 
   - Risk: Features calculated directly from the target variable or its exact mathematical components.
   - Mitigation: Target-defining components and direct outcome proxies (such as `is_below_peer_median`, `ctr`, and `total_clicks`) are explicitly purged from the feature space.

2. Future and Overlapping Windows: 
   - Risk: Features aggregating metrics over time windows that overlap with or follow the prediction label window.
   - Mitigation: Post-event analytics and outcome-window metrics (such as `ga4_sessions`, `engaged_sessions`, and session subsets) are entirely excluded.

3. Product Flags / Existing System Scores: 
   - Risk: Scores or heuristic flags from pre-existing legacy systems that implicitly encode the final decision.
   - Mitigation: Intermediate model scores and historical heuristic targets (`underperformance_score`, `expected_clicks`) are stripped out.

By enforcing a strict exclusion list (`DROP_COLS`), we ensure that no target-corrupted or future-dependent variables enter the training phase.

In [9]:
import pandas as pd
from pathlib import Path

# Define the explicit exclusion list for leakage prevention
DROP_COLS = [
    "client_hash_id",
    "content_hash_id",
    "is_underperforming",
    "underperformance_score",
    "expected_clicks",
    "expected_ctr",
    "is_below_peer_median",
    "pos_bucket",
    "ctr",
    "total_clicks",
    "ga4_sessions",
    "engaged_sessions",
    "scroll_events",
    "sessions_organic",
    "sessions_ai",
    "total_ai_sessions",
    "days_with_ga4",
    "has_ga4",
    "avg_position",
    "total_impressions",
    "days_since_update",
    "freshness_score",
    "optimized_and_fresh",
    "content_maturity",
    "update_optimization_gap",
    "impressions_per_day",
    "lifetime_impressions_est",
    "market_capture_ratio",
    "backlink_efficiency",
]



print("LEAKAGE HUNT VERIFICATION AUDIT")
print("-" * 50)
print(f"Source Loaded: {source_name}")
print(f"Total columns currently in dataset: {df.shape[1]}")

# Test 1: Check how many items from DROP_COLS are still present in the dataframe columns
leaks_found = [col for col in DROP_COLS if col in df.columns]

if len(leaks_found) == 0:
    print("Test 1 Result: Pass. Zero leakage columns found in active dataframe.")
else:
    print(f"Test 1 Result: Warning. The following exclusion columns remain: {leaks_found}")

# Test 2: Specifically test for critical outcome variables that cause target leakage
critical_targets = ["is_below_peer_median", "ctr", "total_clicks", "ga4_sessions"]
active_criticals = [c for c in critical_targets if c in df.columns]

print(f"\nCritical Target/Outcome Variables Remaining: {active_criticals if active_criticals else 'None (Clean - Safe from Target Leakage)'}")

LEAKAGE HUNT VERIFICATION AUDIT
--------------------------------------------------
Source Loaded: Processed Feature Vector
Total columns currently in dataset: 84
Test 1 Result: Warning. The following exclusion columns remain: ['client_hash_id', 'content_hash_id', 'pos_bucket', 'ctr', 'total_clicks', 'ga4_sessions', 'engaged_sessions', 'scroll_events', 'sessions_organic', 'sessions_ai', 'total_ai_sessions', 'days_with_ga4', 'has_ga4', 'avg_position', 'total_impressions', 'days_since_update', 'freshness_score', 'optimized_and_fresh', 'content_maturity', 'update_optimization_gap', 'impressions_per_day', 'lifetime_impressions_est', 'market_capture_ratio', 'backlink_efficiency']

Critical Target/Outcome Variables Remaining: ['ctr', 'total_clicks', 'ga4_sessions']


## 4. What I excluded and why

To maintain data integrity, prevent target leakage, and avoid structural bias, the following fields were explicitly excluded from the model training feature set:

- `client_hash_id`, `content_hash_id`: Excluded from active features to prevent the model from memorizing specific entity identifiers or overfitting to individual accounts.
- `is_underperforming`, `underperformance_score`, `expected_clicks`, `expected_ctr`, `is_below_peer_median`: Excluded because they represent target labels or pre-existing system heuristics, which would cause severe direct target leakage.
- `ctr`, `total_clicks`: Excluded as they are direct components of the outcome window and define the success label.
- `ga4_sessions`, `engaged_sessions`, `scroll_events`, `sessions_organic`, `sessions_ai`, `total_ai_sessions`, `days_with_ga4`, `has_ga4`: Excluded because post-event analytics metrics overlap with the prediction window.
- `avg_position`: Excluded after being transformed into structural `pos_bucket` categories to avoid direct rank-order leakage.
- `total_impressions`: Excluded from raw feature inputs to prevent absolute traffic scale bias after the initial visibility filter (>= 100 impressions)[cite: 1].
- `days_since_update`, `freshness_score`, `optimized_and_fresh`, `content_maturity`, `update_optimization_gap`: Excluded or refined to prevent redundant or overlapping temporal formulations.
- `impressions_per_day`, `lifetime_impressions_est`, `market_capture_ratio`, `backlink_efficiency`: Excluded due to experimental estimation noise or direct overlap with alternative demand metrics.

In [10]:
import pandas as pd
from pathlib import Path

# Comprehensive list of fields excluded from active feature engineering based on pipeline rules
DROP_COLS = [
    "client_hash_id",
    "content_hash_id",
    "is_underperforming",
    "underperformance_score",
    "expected_clicks",
    "expected_ctr",
    "is_below_peer_median",
    "pos_bucket",
    "ctr",
    "total_clicks",
    "ga4_sessions",
    "engaged_sessions",
    "scroll_events",
    "sessions_organic",
    "sessions_ai",
    "total_ai_sessions",
    "days_with_ga4",
    "has_ga4",
    "avg_position",
    "total_impressions",
    "days_since_update",
    "freshness_score",
    "optimized_and_fresh",
    "content_maturity",
    "update_optimization_gap",
    "impressions_per_day",
    "lifetime_impressions_est",
    "market_capture_ratio",
    "backlink_efficiency",
]

# Load dataset to verify alignment with exclusion contract

print("EXCLUDED FIELDS VERIFICATION AUDIT")
print("-" * 50)
print(f"Loaded Source: {source_name}")
print(f"Total fields flagged for exclusion: {len(DROP_COLS)}")

# Check which critical leakage or target columns are absent from active features
critical_targets = ["is_below_peer_median", "ctr", "total_clicks", "ga4_sessions"]
remaining_targets = [col for col in critical_targets if col in df.columns]

print(f"Critical target variables present in active dataset: {remaining_targets if remaining_targets else 'None (Successfully isolated/excluded)'}")

EXCLUDED FIELDS VERIFICATION AUDIT
--------------------------------------------------
Loaded Source: Processed Feature Vector
Total fields flagged for exclusion: 29
Critical target variables present in active dataset: ['ctr', 'total_clicks', 'ga4_sessions']
